In [5]:
import sys
sys.path.insert(0, '../')

In [6]:
import os
import cv2
import numpy as np
import pickle
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from pose_estimation import PoseEstimator

DATA_DIR = 'data'
MODEL_PATH = '../models/shot_classifier.pkl'
CLASSES = ['backhand', 'forehand', 'serve']

## Extract keypoints from labeled images

In [7]:
estimator = PoseEstimator('../models/pose_landmarker.task')

X, y = [], []
skipped = 0

for label, class_name in enumerate(CLASSES):
    folder = os.path.join(DATA_DIR, class_name)
    for fname in os.listdir(folder):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        path = os.path.join(folder, fname)
        frame = cv2.imread(path)
        keypoints = estimator.get_keypoints(frame)
        if keypoints is None:
            os.remove(path)
            skipped += 1
            continue
        X.append(keypoints)
        y.append(label)

X = np.array(X)
y = np.array(y)
print(f'Samples: {len(X)}  |  Skipped and deleted (no pose): {skipped}')
print(f'Class counts: { {CLASSES[i]: int((y == i).sum()) for i in range(len(CLASSES))} }')

I0000 00:00:1778294473.502012 35982546 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
W0000 00:00:1778294473.556047 35982550 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778294473.566109 35982553 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
E0000 00:00:1778294473.572779 35980139 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-05-08T22:54:25.638872-04:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


Samples: 231  |  Skipped and deleted (no pose): 0
Class counts: {'backhand': 83, 'forehand': 93, 'serve': 55}


## Train XGBoost
No testing set only train.

In [8]:
model = XGBClassifier(n_estimators = 75, max_depth = 5, learning_rate = 0.1, eval_metric = 'mlogloss')
model.fit(X, y)

y_pred = model.predict(X)
print(classification_report(y, y_pred, target_names = CLASSES))

              precision    recall  f1-score   support

    backhand       1.00      1.00      1.00        83
    forehand       1.00      1.00      1.00        93
       serve       1.00      1.00      1.00        55

    accuracy                           1.00       231
   macro avg       1.00      1.00      1.00       231
weighted avg       1.00      1.00      1.00       231



## Save model

In [9]:
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(model, f)
print(f'Saved to {MODEL_PATH}')

Saved to ../models/shot_classifier.pkl
